## Import and install dependencies

In [1]:
!pip install evaluate transformers torch accelerate tokenizers sentencepiece -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.0 MB/s eta 0:00:00


In [2]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AutoModel, BertForSequenceClassification


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(device)

cuda


In [3]:
BASE_MODEL_NAME = "prajjwal1/bert-tiny"

num_labels = 2
id2label = {0:'BR', 1:'PT'}
label2id = {'BR':0, 'PT':1}

# model = AutoModelForSequenceClassification.from_pretrained(BASE_MODEL_NAME, num_labels=num_labels, id2label=id2label, label2id=label2id)
model = BertForSequenceClassification.from_pretrained(BASE_MODEL_NAME, num_labels=num_labels, id2label=id2label, label2id=label2id)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/285 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/17.8M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/39 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: prajjwal1/bert-tiny
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were 

In [4]:
model.config

BertConfig {
  "add_cross_attention": false,
  "attention_probs_dropout_prob": 0.1,
  "bos_token_id": null,
  "classifier_dropout": null,
  "dtype": "float32",
  "eos_token_id": null,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 128,
  "id2label": {
    "0": "BR",
    "1": "PT"
  },
  "initializer_range": 0.02,
  "intermediate_size": 512,
  "is_decoder": false,
  "label2id": {
    "BR": 0,
    "PT": 1
  },
  "layer_norm_eps": 1e-12,
  "max_position_embeddings": 512,
  "model_type": "bert",
  "num_attention_heads": 2,
  "num_hidden_layers": 2,
  "pad_token_id": 0,
  "tie_word_embeddings": true,
  "transformers_version": "5.9.0",
  "type_vocab_size": 2,
  "use_cache": true,
  "vocab_size": 30522
}

## Loading Dataset

In [5]:
from datasets import Dataset, DatasetDict, load_dataset, concatenate_datasets

SUBSET_SIZE = 100000

pt_stream = load_dataset('bastao/VeraCruz_PT-BR', "Portugal (PT)", split='train', streaming=True)
ptbr_stream = load_dataset('bastao/VeraCruz_PT-BR', "Brazil (BR)", split='train', streaming=True)
other_stream = load_dataset('bastao/VeraCruz_PT-BR', "Other", split='train', streaming=True)

pt_subset = pt_stream.take(SUBSET_SIZE)
ptbr_subset = ptbr_stream.take(SUBSET_SIZE)
other_subset = other_stream.take(SUBSET_SIZE)

label_map = {"BR": 0, "PT": 1}

def stream_to_dataset(stream, label=None):
    dataset = Dataset.from_list(list(stream))
    if label is not None:
        dataset = dataset.add_column("label", [label] * len(dataset))
    return dataset

def preprocess_labels(example):
    # Replace 'label' with the actual name of your label column
    example["label"] = label_map[example["label"]]
    return example

VERA_CRUZ_DATASETS = DatasetDict({
    "ptbr": stream_to_dataset(ptbr_subset, label_map["BR"]),
    "pt": stream_to_dataset(pt_subset, label_map["PT"]),
    "other": stream_to_dataset(other_subset).map(preprocess_labels),
})

VERA_CRUZ_DATASETS

model.safetensors:   0%|          | 0.00/17.7M [00:00<?, ?B/s]

README.md:   0%|          | 0.00/2.58k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/1903 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/1903 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/1903 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/1903 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/1903 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/1903 [00:00<?, ?it/s]

Map:   0%|          | 0/100000 [00:00<?, ? examples/s]

DatasetDict({
    ptbr: Dataset({
        features: ['text', 'timestamp', 'url', 'source', 'label'],
        num_rows: 100000
    })
    pt: Dataset({
        features: ['text', 'timestamp', 'url', 'source', 'label'],
        num_rows: 100000
    })
    other: Dataset({
        features: ['text', 'timestamp', 'url', 'source', 'label', 'score'],
        num_rows: 100000
    })
})

In [6]:
VERA_CRUZ_DATASETS['ptbr'][0]

{'text': 'Covid-19: Ramiro defende protocolo do Corinthians após polêmica | LANCE!\nCovid-19: Ramiro defende protocolo do Corinthians após polêmica\nO volante comentou sobre o assunto durante entrevista coletiva nesta tarde no CT, juntamente com o companheiro de equipe Gabriel, que falou sobre a final sem torcida\nUma polêmica que tomou conta nos últimos dias foi a recusa do Corinthians em realizar os testes de Covid-19 em seu elenco, antes das finais do Campeonato Paulista. Perguntado sobre o tema em entrevista coletiva juntamente com Gabriel, o volante Ramiro disse que essa questão não irá afetar a preparação da equipe.\n- Esse tipo de questão a gente deixa mais para o extracampo, extra-clube, temos que nos focar no nosso trabalho, ir a campo e fazer um grande jogo. Isso não vai interferir na nossa concentração para esses dois jogos. Estudamos muito a equipe deles, sabemos a forma que a gente tem que se portar para, pós segundo jogo, nos sagrarmos campeões – completou.\nO fato gerou 

## Tokenize

In [7]:
from transformers import AutoTokenizer, BertTokenizerFast

# tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME)
tokenizer = BertTokenizerFast.from_pretrained(BASE_MODEL_NAME)

def preprocess(sample):
    return tokenizer(sample['text'], truncation=True)

# Trunacting max_length=512
TOKENIZED_VERA_CRUZ_DATASETS = VERA_CRUZ_DATASETS.map(
    lambda x: tokenizer(x["text"], truncation=True, max_length=512),
    batched=True
)

TOKENIZED_VERA_CRUZ_DATASETS

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

Map:   0%|          | 0/100000 [00:00<?, ? examples/s]

Map:   0%|          | 0/100000 [00:00<?, ? examples/s]

Map:   0%|          | 0/100000 [00:00<?, ? examples/s]

DatasetDict({
    ptbr: Dataset({
        features: ['text', 'timestamp', 'url', 'source', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 100000
    })
    pt: Dataset({
        features: ['text', 'timestamp', 'url', 'source', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 100000
    })
    other: Dataset({
        features: ['text', 'timestamp', 'url', 'source', 'label', 'score', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 100000
    })
})

In [8]:
TOKENIZED_VERA_CRUZ_DATASETS["ptbr"]

Dataset({
    features: ['text', 'timestamp', 'url', 'source', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 100000
})

## Fine-tuning

In [9]:

CONCATENATED_DATASETS = concatenate_datasets([
    TOKENIZED_VERA_CRUZ_DATASETS['pt'],
    TOKENIZED_VERA_CRUZ_DATASETS['ptbr']
]).shuffle(seed=42)

TRAIN_SIZE = int(0.8 * len(CONCATENATED_DATASETS))
TRAIN_DATASETS = CONCATENATED_DATASETS.select(range(TRAIN_SIZE))
EVAL_DATASETS = CONCATENATED_DATASETS.select(range(TRAIN_SIZE, len(CONCATENATED_DATASETS)))


In [10]:
from sklearn.metrics import accuracy_score
from transformers import TrainingArguments, Trainer, DataCollatorWithPadding

def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    return {"accuracy": accuracy_score(labels, preds)}

# Hyperparameters from the original https://huggingface.co/bastao/PeroVaz_PT-BR_Classifier
params = {
    "learning_rate": 5e-05,
    "train_batch_size": 256,
    "eval_batch_size": 256,
    "seed": 42,
    "lr_scheduler_type": "linear",
    "max_steps": 2500,
    "fp16": True, # Native AMP
}

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

training_args = TrainingArguments(
    output_dir='outputs',
    per_device_train_batch_size=params["train_batch_size"],
    per_device_eval_batch_size=params["eval_batch_size"],
    learning_rate=params["learning_rate"],
    max_steps=params["max_steps"],
    lr_scheduler_type=params["lr_scheduler_type"],
    seed=params["seed"],
    fp16=params["fp16"],
    logging_strategy='epoch',
    eval_strategy='epoch',
    logging_steps=50,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=TRAIN_DATASETS,
    eval_dataset=EVAL_DATASETS,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)


In [11]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,0.447364,0.326071,0.863675
2,0.310678,0.293897,0.875650
3,0.283138,0.276326,0.883050
4,0.270524,0.276365,0.882875


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=2500, training_loss=0.327925927734375, metrics={'train_runtime': 1200.7202, 'train_samples_per_second': 533.013, 'train_steps_per_second': 2.082, 'total_flos': 813111705600000.0, 'train_loss': 0.327925927734375, 'epoch': 4.0})

## Evaluation

In [15]:
from transformers import pipeline

classifier = pipeline("text-classification", model=model, tokenizer=tokenizer, device=device)
text = "Isso é um texto em portugês brasileiro."
result = classifier(text)

print(f"Text: {text}")
print(f"Result: {result}")

Text: Isso é um texto em portugês brasileiro.
Result: [{'label': 'BR', 'score': 0.6415908336639404}]


In [13]:
import pandas as pd

# Extract logs from trainer state
history = trainer.state.log_history

# Filter for logs that contain both training and evaluation metrics
# Since we set eval_strategy='epoch', we expect logs at the end of each epoch
df_logs = pd.DataFrame(history)

# We want to align training loss and evaluation metrics
# Training logs usually have 'loss', 'epoch', 'step'
# Evaluation logs have 'eval_loss', 'eval_accuracy', 'epoch', 'step'
train_logs = df_logs[df_logs['loss'].notna()][['epoch', 'step', 'loss']]
eval_logs = df_logs[df_logs['eval_loss'].notna()][['epoch', 'step', 'eval_loss', 'eval_accuracy']]

results = pd.merge(train_logs, eval_logs, on=['epoch', 'step'])
results.columns = ['Epoch', 'Step', 'Training Loss', 'Validation Loss', 'Accuracy']

print(results.to_string(index=False))

 Epoch  Step  Training Loss  Validation Loss  Accuracy
   1.0   625       0.447364         0.326071  0.863675
   2.0  1250       0.310678         0.293897  0.875650
   3.0  1875       0.283138         0.276326  0.883050
   4.0  2500       0.270524         0.276365  0.882875
